# Notebook 16 — FastAPI Serving & End-to-End Demo

**Scope:** Offline RetailRocket e-commerce recommendation proxy, **not real ads CTR or ad auctions**. This demo serves a frozen September 1 snapshot from Notebook 08 Hybrid Top100 candidates. It runs **actual LightGBM inference → Top20 gate → actual DeepFM inference → optional Item2Vec MMR Top10** on each request.

**Important boundaries:** candidate retrieval and historical feature computation are offline/materialized; the API does not perform online retrieval, feature-store reads, retraining, or online event ingestion. It is a *serving demo*, not a production system.

**Run from:** `C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks` (or project root). Notebook writes `serving/app.py`, `serving/demo_candidates.csv`, and `reports/notebook16_serving_demo.json`. Do not run `uvicorn` in a blocking notebook cell.

**Dependencies:** `pip install fastapi uvicorn pydantic pandas numpy torch lightgbm scikit-learn gensim scipy requests`.

In [1]:
from pathlib import Path
import json, time, pickle, importlib.util
import numpy as np
import pandas as pd

ROOT=Path.cwd().resolve()
if not (ROOT/'data'/'processed').is_dir(): ROOT=ROOT.parent
assert (ROOT/'data'/'processed').is_dir(), 'Run notebook from project root or notebooks/'
PROC=ROOT/'data'/'processed'; REPORTS=ROOT/'reports'; MODELS=ROOT/'models'; SERVING=ROOT/'serving'
REPORTS.mkdir(exist_ok=True); SERVING.mkdir(exist_ok=True)
required=[PROC/'notebook08_test_hybrid_candidates.csv',PROC/'test_history.csv',
          PROC/'notebook10_test_scored_candidates.csv',PROC/'notebook09_test_scored_candidates.csv',
          PROC/'notebook13_test_cascade_top10.csv',REPORTS/'notebook09_lightgbm_preranker.pkl',
          REPORTS/'notebook10_deepfm_model.pt',MODELS/'notebook06_test_item2vec.model']
missing=[str(p) for p in required if not p.exists()]
assert not missing, f'Missing artifacts: {missing}'
print('Project root:',ROOT)

Project root: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks


## 1. Build an offline feature snapshot for a small, reproducible serving cohort

All features come from **pre-September-1 history**. We materialize only **100 users × 100 candidates** to keep local startup light. User IDs are chosen deterministically from the frozen September test cohort. Unknown users return HTTP 404; do not silently pretend this is a cold-start service.

In [2]:
FEATURES=['retrieval_rank','rrf_inverse_rank','from_itemcf','from_faiss_fallback','from_popularity',
          'source_count','log_item_views','log_item_unique_viewers','log_user_views',
          'log_user_unique_items','user_last_event_days','item_last_view_days',
          'item_view_share','log_item_carts','log_item_purchases']
def make_features(candidates,history,cutoff):
    views=history.loc[history.event.eq('view'),['visitorid','itemid','timestamp']]
    stats=views.groupby('itemid',sort=False).agg(item_views=('visitorid','size'),item_unique_viewers=('visitorid','nunique'),item_last_view=('timestamp','max'))
    users=views.groupby('visitorid',sort=False).agg(user_views=('itemid','size'),user_unique_items=('itemid','nunique'),user_last_event=('timestamp','max'))
    other=history.loc[history.event.isin(['addtocart','transaction']),['itemid','event']]
    if len(other):
        counts=other.groupby(['itemid','event']).size().unstack(fill_value=0)
        stats['item_carts']=counts['addtocart'] if 'addtocart' in counts else 0
        stats['item_purchases']=counts['transaction'] if 'transaction' in counts else 0
    else: stats['item_carts']=0;stats['item_purchases']=0
    stats[['item_carts','item_purchases']]=stats[['item_carts','item_purchases']].fillna(0)
    d=candidates.merge(stats,left_on='itemid',right_index=True,how='left',validate='many_to_one')
    d=d.merge(users,left_on='visitorid',right_index=True,how='left',validate='many_to_one')
    ms=int(cutoff.timestamp()*1000)
    d['retrieval_rank']=d['rank'].astype('float32')
    d['rrf_inverse_rank']=(1.0/(60+d['rank'])).astype('float32')
    d['source_count']=d[['from_itemcf','from_faiss_fallback','from_popularity']].sum(axis=1)
    for src,dst in [('item_views','log_item_views'),('item_unique_viewers','log_item_unique_viewers'),('user_views','log_user_views'),('user_unique_items','log_user_unique_items'),('item_carts','log_item_carts'),('item_purchases','log_item_purchases')]:
        d[dst]=np.log1p(d[src].fillna(0)).astype('float32')
    d['user_last_event_days']=((ms-d.user_last_event)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_last_view_days']=((ms-d.item_last_view)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_view_share']=(d.item_views.fillna(0)/max(1,len(views))).astype('float32')
    for name in FEATURES:d[name]=d[name].fillna(0).astype('float32')
    return d[['visitorid','itemid','rank']+FEATURES].copy()

raw=pd.read_csv(PROC/'notebook08_test_hybrid_candidates.csv')
assert raw.groupby('visitorid').size().eq(100).all()
users=np.sort(raw.visitorid.unique())
DEMO_USERS=min(100,len(users))
selected_users=np.random.default_rng(42).choice(users,size=DEMO_USERS,replace=False)
selected=raw.loc[raw.visitorid.isin(selected_users)].copy()
history=pd.read_csv(PROC/'test_history.csv',usecols=['timestamp','visitorid','itemid','event'])
cutoff=pd.Timestamp('2015-09-01',tz='UTC')
assert int(history.timestamp.max())<int(cutoff.timestamp()*1000)
start=time.perf_counter()
snapshot=make_features(selected,history,cutoff)
build_s=time.perf_counter()-start
assert len(snapshot)==DEMO_USERS*100 and snapshot[FEATURES].notna().all().all()
snapshot.to_csv(SERVING/'demo_candidates.csv',index=False)
print('Serving cohort:',DEMO_USERS,'users,',len(snapshot),'candidate rows')
print('Offline snapshot build seconds:',round(build_s,3))

Serving cohort: 100 users, 10000 candidate rows
Offline snapshot build seconds: 1.39


## 2. Write the actual FastAPI application

The app loads trained models once at startup, preserves model feature order, uses `StandardScaler.transform()` from Notebook 10, and scores only **Top20** with DeepFM. MMR uses the same formula and split-specific Word2Vec vectors as Notebook 13. Scores are **ranking scores, not calibrated CTR/CVR**.

In [3]:
APP_SOURCE = '"""Offline RetailRocket ranking-serving demo; not a live ad or CTR prediction system."""\nfrom pathlib import Path\nimport os, pickle, time\nfrom functools import lru_cache\nfrom threading import Lock\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom sklearn.preprocessing import StandardScaler\nfrom gensim.models import Word2Vec\nfrom fastapi import FastAPI, HTTPException, Query\n\nROOT=Path(os.environ.get(\'RECSYS_ROOT\', Path(__file__).resolve().parent.parent)).resolve()\nSERVING=ROOT/\'serving\'; REPORTS=ROOT/\'reports\'; MODELS=ROOT/\'models\'\nTOP_N=20; FINAL_K=10; MMR_LAMBDA=.9\n\nclass NumericDeepFM(nn.Module):\n    def __init__(self,n_fields,embed_dim=8,hidden=(64,32),dropout=.2):\n        super().__init__();self.linear=nn.Linear(n_fields,1)\n        self.fm_vectors=nn.Parameter(torch.randn(n_fields,embed_dim)*.02)\n        layers=[];d=n_fields*embed_dim\n        for width in hidden:\n            layers.extend([nn.Linear(d,width),nn.ReLU(),nn.Dropout(dropout)]);d=width\n        layers.append(nn.Linear(d,1));self.mlp=nn.Sequential(*layers)\n    def forward(self,x):\n        e=x.unsqueeze(-1)*self.fm_vectors.unsqueeze(0);s=e.sum(dim=1)\n        fm=.5*(s.square()-(e.square()).sum(dim=1)).sum(dim=1,keepdim=True)\n        return (self.linear(x)+fm+self.mlp(e.flatten(start_dim=1))).squeeze(-1)\n\ndef mmr_order(items,scores,wv,lam=.9,k=10):\n    items=np.asarray(items,dtype=np.int64);scores=np.asarray(scores,dtype=np.float64)\n    lo,hi=float(scores.min()),float(scores.max())\n    rel=(scores-lo)/(hi-lo) if hi>lo else np.zeros(len(scores),dtype=np.float64)\n    vectors=np.zeros((len(items),wv.vector_size),dtype=np.float32)\n    for j,item in enumerate(items):\n        key=str(int(item))\n        if key in wv: vectors[j]=wv[key]\n    norms=np.linalg.norm(vectors,axis=1,keepdims=True)\n    np.divide(vectors,np.maximum(norms,1e-12),out=vectors)\n    sim=np.clip(vectors@vectors.T,0,1);np.fill_diagonal(sim,0)\n    remaining=np.ones(len(items),dtype=bool);redundancy=np.zeros(len(items));chosen=[]\n    for _ in range(k):\n        scores_mmr=lam*rel-(1-lam)*redundancy\n        scores_mmr[~remaining]=-np.inf\n        j=int(np.argmax(scores_mmr));chosen.append(j);remaining[j]=False\n        redundancy=np.maximum(redundancy,sim[:,j])\n    return np.asarray(chosen,dtype=np.int64)\n\nclass RankingService:\n    def __init__(self):\n        with open(REPORTS/\'notebook09_lightgbm_preranker.pkl\',\'rb\') as f:\n            pkg=pickle.load(f)\n        self.lgb=pkg[\'model\'];self.features=list(pkg[\'features\'])\n        try: p=torch.load(REPORTS/\'notebook10_deepfm_model.pt\',map_location=\'cpu\',weights_only=True)\n        except TypeError: p=torch.load(REPORTS/\'notebook10_deepfm_model.pt\',map_location=\'cpu\')\n        assert self.features==list(p[\'features\']), \'Feature ordering mismatch\'\n        self.deepfm=NumericDeepFM(p[\'n_fields\'],p[\'embed_dim\'],p[\'hidden\'],p[\'dropout\'])\n        self.deepfm.load_state_dict(p[\'state_dict\']);self.deepfm.eval()\n        self.scaler=StandardScaler()\n        self.scaler.mean_=np.asarray(p[\'scaler_mean\'],dtype=np.float64)\n        self.scaler.scale_=np.asarray(p[\'scaler_scale\'],dtype=np.float64)\n        self.scaler.var_=self.scaler.scale_**2\n        self.scaler.n_features_in_=len(self.features)\n        self.wv=Word2Vec.load(str(MODELS/\'notebook06_test_item2vec.model\')).wv\n        frame=pd.read_csv(SERVING/\'demo_candidates.csv\')\n        assert frame.groupby(\'visitorid\').size().eq(100).all()\n        assert not frame.duplicated([\'visitorid\',\'itemid\']).any()\n        self.candidates={int(u):g.sort_values([\'rank\',\'itemid\'],kind=\'mergesort\').reset_index(drop=True)\n                         for u,g in frame.groupby(\'visitorid\',sort=False)}\n        self.lock=Lock() # avoid cross-request model CPU thread contention in this demo\n    def recommend(self,visitorid:int,k:int=10,mmr:bool=True):\n        if visitorid not in self.candidates: raise KeyError(visitorid)\n        g=self.candidates[visitorid]\n        x=g[self.features].astype(\'float32\') # preserve feature names for LightGBM\n        a=time.perf_counter_ns()\n        with self.lock:\n            ls=self.lgb.predict_proba(x)[:,1]\n            b=time.perf_counter_ns()\n            # Match Notebook 09: descending score, ascending retrieval rank, then item ID\n            gate=np.lexsort((g.itemid.to_numpy(),g[\'rank\'].to_numpy(),-ls))[:TOP_N]\n            c=time.perf_counter_ns()\n            z=np.clip(self.scaler.transform(x.iloc[gate].to_numpy(dtype=np.float32)),-8,8).astype(\'float32\')\n            with torch.inference_mode(): ds=self.deepfm(torch.from_numpy(z)).numpy()\n            d=time.perf_counter_ns()\n            gated_items=g.itemid.to_numpy()[gate]\n            # Match Notebook 13: deepfm_score descending, lgb_rank ascending, item ID\n            ordered=np.lexsort((gated_items,np.arange(1,TOP_N+1),-ds))\n            ordered_items=gated_items[ordered];ordered_scores=ds[ordered]\n            if mmr: idx=mmr_order(ordered_items,ordered_scores,self.wv,MMR_LAMBDA,k)\n            else: idx=np.arange(k)\n            e=time.perf_counter_ns()\n        recs=[{\'rank\':i+1,\'itemid\':int(ordered_items[j]),\'ranking_score\':float(ordered_scores[j])}\n              for i,j in enumerate(idx)]\n        return {\'visitorid\':visitorid,\'k\':k,\'mmr\':mmr,\'snapshot_cutoff\':\'2015-09-01\',\n                \'candidate_pool\':100,\'deepfm_candidates\':TOP_N,\'recommendations\':recs,\n                \'latency_ms\':{\'lightgbm\':(b-a)/1e6,\'gate\':(c-b)/1e6,\n                              \'deepfm_and_scaler\':(d-c)/1e6,\'sort_and_mmr\':(e-d)/1e6,\n                              \'ranking_total\':(e-a)/1e6},\n                \'limitations\':\'Precomputed candidates/features; ranking scores are not calibrated CTR.\'}\n\n@lru_cache(maxsize=1)\ndef get_service(): return RankingService()\n\napp=FastAPI(title=\'RetailRocket Multi-Stage Ranking Demo\',version=\'1.0.0\',\n            description=\'Frozen-snapshot recommendation proxy; not an online ads auction system.\')\n\n@app.get(\'/health\')\ndef health():\n    s=get_service()\n    return {\'status\':\'ok\',\'loaded_users\':len(s.candidates),\'candidate_pool\':100,\'deepfm_gate\':TOP_N,\n            \'snapshot_cutoff\':\'2015-09-01\'}\n\n@app.get(\'/demo-users\')\ndef demo_users(limit:int=Query(default=10,ge=1,le=100)):\n    return {\'visitorids\':sorted(get_service().candidates)[:limit]}\n\n@app.get(\'/recommend/{visitorid}\')\ndef recommend(visitorid:int,k:int=Query(default=10,ge=1,le=10),mmr:bool=True):\n    try: return get_service().recommend(visitorid,k=k,mmr=mmr)\n    except KeyError: raise HTTPException(status_code=404,detail=\'User not in frozen demo cohort\')\n'
app_path=SERVING/'app.py'
app_path.write_text(APP_SOURCE,encoding='utf-8')
print('Wrote:',app_path)

Wrote: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\serving\app.py


## 3. Import and smoke-test the actual application (no server needed)

Compare the full Top10 (with MMR) to Notebook 13's frozen offline replay. A mismatch is a **failed validation**; do not silently accept it. We also check the pre-MMR order against Notebook 10/09 scores to help diagnose any discrepancy.

In [5]:
spec=importlib.util.spec_from_file_location('rr_demo_app',SERVING/'app.py')
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
service=module.get_service()
assert len(service.candidates)==DEMO_USERS
assert list(service.features)==FEATURES
ref=pd.read_csv(PROC/'notebook13_test_cascade_top10.csv',usecols=['visitorid','itemid','final_rank'])
reference={int(u):g.sort_values('final_rank').itemid.astype('int64').tolist()
           for u,g in ref.loc[ref.visitorid.isin(selected_users)].groupby('visitorid')}
checked=0;mismatches=[];latencies=[]
for uid in sorted(service.candidates):
    response=service.recommend(uid,10,True)
    actual=[x['itemid'] for x in response['recommendations']]
    expected=reference[uid]
    if actual!=expected: mismatches.append({'visitorid':uid,'actual':actual,'expected':expected})
    latencies.append(response['latency_ms']['ranking_total']);checked+=1
print('Checked users:',checked,'Exact Top10 matches:',checked-len(mismatches),'Mismatches:',len(mismatches))
if mismatches: print('First mismatch:',mismatches[0])
assert not mismatches, 'Online model inference does not exactly replay Notebook13 Top10; inspect first mismatch above'
print('PASS: actual conditional inference + MMR matches frozen Notebook13 Top10')
print('Direct-call P50/P95 ms (including scaler and MMR):',np.percentile(latencies,[50,95]))

Checked users: 100 Exact Top10 matches: 100 Mismatches: 0
PASS: actual conditional inference + MMR matches frozen Notebook13 Top10
Direct-call P50/P95 ms (including scaler and MMR): [3.9868  5.55543]


## 4. HTTP API smoke test and JSON report

Uses FastAPI's `TestClient` (requires `httpx`: `pip install httpx`). The actual server command appears below. **TestClient timing is not production HTTP latency.**

In [6]:
from fastapi.testclient import TestClient
client=TestClient(module.app)
health=client.get('/health');assert health.status_code==200
uid=int(sorted(service.candidates)[0])
resp=client.get(f'/recommend/{uid}',params={'k':10,'mmr':'true'})
assert resp.status_code==200,resp.text
payload=resp.json()
assert len(payload['recommendations'])==10
assert client.get('/recommend/-99999999').status_code==404
assert client.get(f'/recommend/{uid}',params={'k':11}).status_code==422
assert client.get('/demo-users',params={'limit':5}).status_code==200
print('HTTP smoke tests: PASS')
print('Example request: GET /recommend/'+str(uid)+'?k=10&mmr=true')
print('First three results:',payload['recommendations'][:3])

report={'method':'FastAPI frozen-snapshot serving demo with real LightGBM and conditional DeepFM inference',
        'snapshot_cutoff':'2015-09-01','demo_users':DEMO_USERS,'candidates_per_user':100,
        'deepfm_gate':20,'mmr_lambda':.9,'offline_snapshot_build_seconds':build_s,
        'replay_users_checked':checked,'exact_top10_replay_matches':checked-len(mismatches),
        'http_smoke_tests_passed':True,'direct_call_ranking_with_mmr_p50_ms':float(np.percentile(latencies,50)),
        'direct_call_ranking_with_mmr_p95_ms':float(np.percentile(latencies,95)),
        'scope_excluded':['online retrieval','online historical feature computation','network and concurrency load test',
                          'production monitoring','calibrated CTR or auction modeling']}
path=REPORTS/'notebook16_serving_demo.json'
path.write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Saved:',path)

HTTP smoke tests: PASS
Example request: GET /recommend/65029?k=10&mmr=true
First three results: [{'rank': 1, 'itemid': 122350, 'ranking_score': 0.08365138620138168}, {'rank': 2, 'itemid': 48011, 'ranking_score': -0.3173029124736786}, {'rank': 3, 'itemid': 219512, 'ranking_score': -1.8929930925369263}]
Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook16_serving_demo.json


c:\Users\wyr95\.conda\envs\torch\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 5. Run the API locally (Windows)

Open a **new Anaconda Prompt** in the same `torch` environment, from your project root:

```powershell
cd C:\YR_Python_Code\multi-stage-ads-recommendation
conda activate torch
pip install fastapi uvicorn httpx
python -m uvicorn serving.app:app --host 127.0.0.1 --port 8000
```

Open **http://127.0.0.1:8000/docs** for interactive Swagger docs, then `GET /demo-users` and `GET /recommend/{visitorid}`. Example from a second terminal:

```powershell
curl http://127.0.0.1:8000/health
curl http://127.0.0.1:8000/demo-users
```

**Performance interpretation:** Notebook 15 is the controlled inference-only CPU benchmark. Notebook 16 adds scaler, MMR and in-process service overhead, but still excludes candidate retrieval and historical feature computation. Neither number is production p95.

**Next step:** Upload `reports/notebook16_serving_demo.json`. If you want a portfolio-ready deliverable after that, we can generate a project README and architecture diagram with validated results.